# Starting and Joining Threads

You will run independent Java tasks concurrently and use completion checks to report their results correctly.

You will describe separate jobs, give each its own worker, and decide when the caller may report their results. The examples use finite loops with small inputs, so you can trace their counts without guessing which worker finishes first.

## Learning Goals

- Start two Runnable jobs and join both before reporting their final results.
- Distinguish direct run from start and explain which output ordering is guaranteed.

## Why This Matters

Programs often have separate jobs whose progress overlaps. A report still needs a clear point at which every required result is ready. This lesson separates the ability to make concurrent progress from a promise of simultaneous execution or speed. That distinction lets you explain what a program guarantees, rather than treating one observed schedule as its specification.

In larger applications, separate jobs can prepare independent parts of a report while the caller coordinates their completion. Without a clear completion rule, the report can read unfinished work. This lesson extends your object and interface skills with that coordination rule; the next lesson considers the different problem of workers changing shared state.

## Check Your Starting Point

Recall how two object references can identify different objects with their own fields. Explain what implementing an interface promises and what an ordinary method call does.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="starting-point-interpretation"></a>

References identify objects; separate work objects can own separate result fields. An interface fixes required behavior; an ordinary call runs on its caller.

</details>

## Video Demonstration

Watch how separate work objects, two starts, and two joins lead to a completed report.

<video controls preload="metadata" width="960" aria-label="Starting and joining Java threads demonstration">
<source src="media/01_starting_and_joining_threads/demo.mp4" type="video/mp4">
<track kind="captions" src="media/01_starting_and_joining_threads/captions.vtt" srclang="en" label="English">
</video>

[Read the starting and joining threads video transcript.](media/01_starting_and_joining_threads/transcript.md)

## Concept

### Let separate jobs make progress

A campus welcome desk has two preparation jobs. One counts three badge checks; the other counts two supply checks. In our small model, each loop iteration adds one completed check. The program does not inspect actual badges or read files. We want the caller to report both finished counts accurately.

**Concurrency** means that separate activities can make progress during the same period. A processor can switch between activities, or different processors can run them at the same time. **Parallel execution** is the second case: actual simultaneous execution. These terms describe different guarantees.

Starting two Java workers allows concurrent progress. It does not promise that both execute at the same instant, that they finish in a particular order, or that this tiny program runs faster. Our learning goal is to arrange the work and recognize when its results are ready. We will measure the completed counts, not speed.

### Give the work and its worker separate roles

A **Runnable** is a Java interface whose `run` method describes work to perform. The class below uses familiar interface and object-state syntax. It is the work description from the complete example later in the lesson.

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
```

The keyword **`implements`** states that `CountJob` supplies the behavior required by `Runnable`. The `@Override` annotation lets the compiler check that `run` implements the inherited operation. The constructor records how many checks this job should count and initializes its result to zero. The loop increases `total` once for each index below `limit`. The getter returns the current count; it does not wait for work to finish.

A **Thread** supplies a path of execution for that work. In the following fragment, `first` refers to the job and `firstWorker` refers to the thread prepared to execute it. This fragment assumes the `CountJob` declaration above.

```java
CountJob first = new CountJob(3);
Thread firstWorker = new Thread(first);
```

The first construction creates the job's state. The second construction associates that same job with a worker. Neither line starts its loop on a new thread. Keeping those roles separate will help us tell the difference between having a work object and requesting its execution.

### Request execution with start

Once the job and thread exist, the caller can request execution:

```java
firstWorker.start();
```

The **`start`** method requests that the worker execute its work. The caller can continue with its own next statement while the worker makes progress. Returning from `start` does not mean that the job has finished. We will use a separate operation for that question.

Calling **`run`** directly has a different meaning. The expression `first.run()` is an ordinary method call. It executes the counting loop on the thread that made that call, just as earlier method calls did. Naming the method `run` does not create a new execution path.

This distinction solves a practical problem: a caller that needs separate work to proceed must request worker execution, not merely invoke the work itself. The support example later identifies the thread making each call so you can check the distinction. For now, remember the three separate actions: construct the job, construct its thread, and start that thread.

### Use a new thread for a new execution

A **thread lifecycle** describes the stages a thread passes through. Our worker is first constructed, then started, and eventually terminated when its work returns. A `Thread` object can be started only once.

Calling `start` again on a thread that has already been started is invalid, even if its earlier work has finished. Waiting for termination does not reset that thread. If the application needs another independent work session, it constructs a fresh job and a fresh thread.

This rule also matters when running notebook examples again. Each complete example below creates new work objects and threads, so a complete rerun begins with fresh counts and valid worker lifecycles. Running only a later `start` statement against an old thread would be a different operation. The later diagnostic shows why a second start is rejected.

The method **`isAlive`** reports whether a thread has started and has not yet terminated. The final example uses it after both joins to check that neither worker remains alive. A value of `false` by itself does not distinguish a never-started thread from a terminated one; the preceding starts and joins supply that context.

### Separate worker order from report order

Each counting worker follows its own loop in order. Across workers, their actions may be mixed. **Interleaving** is an ordering in which actions from different threads occur among one another. One worker could finish all its checks first, or the workers could take turns making progress.

Our program will not print a line from inside either counting loop. Instead, the caller will wait for both workers and then print the first job's total followed by the second job's total. That report order is determined by the caller's statements.

A report that lists `First` before `Second` therefore does not prove that the first worker finished first. It tells us which result the caller displayed first. This separation makes the report repeatable while allowing the workers' execution order to vary. It also prevents one observed run from being mistaken for a scheduling guarantee.

### Wait before reading the final results

The caller needs a dependable point at which a job's result is complete. The **`join`** method supplies that point for a specified thread. In our example, both workers are started before either join:

```java
firstWorker.start();
secondWorker.start();
firstWorker.join();
secondWorker.join();
```

These lines are a fragment from the complete program and assume both worker variables have been constructed. The first two lines request both executions. The caller then waits for the first worker and afterward waits for the second. The second worker can continue making progress while the caller waits for the first; joining the first does not pause the second.

Here each join has no timeout and targets a worker that was started. When it returns normally, that worker has terminated. Its earlier writes are also **visible** to the caller: the caller can read the values that worker stored before it finished. This is why reading `first.getTotal()` after the first join can retrieve its finished count.

One join concerns one target. The caller must also join the second worker before treating its count as final. Reading both totals immediately after the starts would leave completion unestablished, even if a particular run happened to finish quickly.

A join can report the checked `InterruptedException` instead of returning normally. That exit does not establish that its target finished. If a join in a notebook cell is interrupted, the cell can report that exception instead of reaching its final print statements. Later material teaches coordinated responses to interruption. Our expected report describes the normal path where both joins complete.

### Keep each worker's changing state separate

**Worker confinement** is an ownership rule: a worker changes its own state, and other code waits for completion before inspecting that state. Each `CountJob` in this lesson has a separate `total` field in a separate object. The first worker changes the first job's total; the second changes the second job's total.

The caller follows the other half of the rule. It reads those totals only after the corresponding joins establish termination and visibility. The getter itself is not a completion check. Its safe place in this example comes from the surrounding lifecycle and ownership decisions.

The keyword `private` limits which source code can access a field directly. It does not, by itself, prevent two threads from calling methods on the same object. Passing the same `CountJob` to two workers would change the ownership arrangement and would need different reasoning.

We now have the pieces for the complete example: separate work objects, separate worker threads, both starts, both joins, and a report made by the caller. The next lesson changes one key assumption by giving workers a shared counter and teaching how to protect its updates.

## Worked Example

A campus welcome desk models three badge checks and two supply checks as separate counting jobs. The caller waits for both completed counts before reporting them.

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(3);
CountJob second = new CountJob(2);
Thread firstWorker = new Thread(first);
Thread secondWorker = new Thread(second);
firstWorker.start();
secondWorker.start();
firstWorker.join();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));
```

Expected output:

```text
First: 3
Second: 2
Workers alive: false
```

The complete program creates two CountJob objects so that each preparation job starts with its own zero total. The first receives a limit of three and the second a limit of two. Each Thread is associated with its corresponding job. The caller starts both workers before waiting for either, so both execution requests exist before the first join.

After the two joins return normally, the caller reads the finished totals and asks whether either worker remains alive. The loop limits account for First: 3 and Second: 2. The final Workers alive: false report is meaningful because both workers were started and then joined. Its value alone would not distinguish termination from a thread that never started.

The report comes from the caller in a fixed order. It does not record the order in which the jobs finished. Use this completed run to connect each displayed value to the job that produced it, then retrieve that reasoning in the guided task.

## Guided Practice

Use the explained program first to retrieve its reasoning, then apply the ideas to distinct completion, modification and debugging tasks.

Recall the already demonstrated result and explain how the program produces it. Identify the relevant owned or shared state and the rule that allows its final observation. Then run the complete example and preserve this response; this is retrieval, not an unseen prediction.

In [ ]:
Your response:


In [ ]:
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(3);
CountJob second = new CountJob(2);
Thread firstWorker = new Thread(first);
Thread secondWorker = new Thread(second);
firstWorker.start();
secondWorker.start();
firstWorker.join();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));

Record the actual output from this run. Compare them with the already explained result. Identify any difference without changing your original retrieval response.

In [ ]:
Your response:


Trace each worker’s private total and the caller’s two joins. Explain completion and visibility, and distinguish print order from finish order.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="worked-interpretation"></a>

The first job's own limit explains its total, and the second job's limit explains its separate total. Each normal join establishes its target's completion and prior-write visibility. The caller's final print order does not establish worker finish order.

</details>

<details id="animation-owned_workers_and_join" class="animation-panel" open>
<summary>Own separate results; join before reporting — show or hide animation</summary>
<p><img src="media/01_starting_and_joining_threads/owned_workers_and_join.gif" alt="Separate jobs begin with totals zero. Each worker starts its own job. Normal joins establish the completed totals three and two. The caller prints First: 3, Second: 2, and Workers alive: false." width="960" style="max-width:100%;height:auto;"></p>
</details>

The sequence separates requesting work from establishing completion. Each normal join concerns one started worker. The caller reads both results after both waits. Its print order does not reveal the workers’ finish order. This silent loop lasts 25 seconds. Hide the animation to remove visible motion.

[View this final state as a still image](media/01_starting_and_joining_threads/owned_workers_and_join_still.png).


<a id="animation-owned_workers_and_join-still"></a>

[View the final state as a still image](media/01_starting_and_joining_threads/owned_workers_and_join_still.png).


### Identify the thread executing the work

**`Thread.currentThread()`** returns the Thread object for the execution path making the call. Store that reference as `caller` before creating the work. The identity comparison `==` asks whether two references identify the same Thread object.

This probe reuses lambda expressions and captured references from the earlier lesson. Because Runnable requires a `run` method with no parameters, `()` is the lambda's empty parameter list. The expression after the arrow is the work to perform later. The lambda captures `caller`, which is not reassigned. Read the two invocation paths before running them.

Read the complete caller-context probe below without running it. Predict each caller-identity comparison and explain the distinction between run and start.

In [ ]:
Your response:


In [ ]:
Thread caller = Thread.currentThread();
Runnable compare = () -> System.out.println("On caller: " + (Thread.currentThread() == caller));
compare.run();
Thread worker = new Thread(compare);
worker.start();
worker.join();

Record both actual identity reports and explain which execution path makes each call. Explain what the normal join establishes before the caller continues.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="support-direct-run-answer-source-1"></a>

```java
Thread caller = Thread.currentThread();
Runnable compare = () -> System.out.println("On caller: " + (Thread.currentThread() == caller));
compare.run();
Thread worker = new Thread(compare);
worker.start();
worker.join();
```

Expected output:

```text
On caller: true
On caller: false
```

<a id="support-direct-run-interpretation"></a>

Direct run executes on the caller; start requests execution on the other Thread. The identity comparison therefore differs. Join the started worker before inspecting its completed report.

<details id="animation-direct_run_vs_start" class="animation-panel" open>
<summary>Distinguish direct run from worker execution — show or hide animation</summary>
<p><img src="media/01_starting_and_joining_threads/direct_run_vs_start.gif" alt="The caller identity is captured by one Runnable. A direct run call prints On caller: true. A separate started Thread invokes the same work and prints On caller: false. A normal join establishes worker completion." width="960" style="max-width:100%;height:auto;"></p>
</details>

Creating the lambda does not invoke its body. The direct call invokes that body on the caller; starting the associated Thread invokes it on the worker. The two comparisons therefore produce different results. The final join waits for the started worker. This silent loop lasts 19 seconds. Hide the animation to remove visible motion.


<a id="animation-direct_run_vs_start-still"></a>

[View the final state as a still image](media/01_starting_and_joining_threads/direct_run_vs_start_still.png).


</details>

### Check the one-start lifecycle

The next work value uses `() -> {}`: an empty parameter list and an empty body. The work returns without performing any task. This lets the example focus on the Thread lifecycle.

**`IllegalThreadStateException`** identifies an operation that is invalid for a thread's lifecycle state. The supplied catch handles that exception when the second start is attempted. Read the entire program before running it.

Before running, identify the Thread object that the second start targets. Recall the single-start rule and predict the caught diagnostic category.

In [ ]:
Your response:


In [ ]:
Runnable work = () -> {};
Thread worker = new Thread(work);
worker.start();
worker.join();
try {
    worker.start();
} catch (IllegalThreadStateException problem) {
    System.out.println("Create a new Thread for another start.");
}

Record the actual caught diagnostic report. Explain why constructing a new work object alone would not reset the already started Thread.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="support-restart-answer-source-1"></a>

```java
Runnable work = () -> {};
Thread worker = new Thread(work);
worker.start();
worker.join();
try {
    worker.start();
} catch (IllegalThreadStateException problem) {
    System.out.println("Create a new Thread for another start.");
}
```

Expected output:

```text
Create a new Thread for another start.
```

<a id="support-restart-interpretation"></a>

A Thread object has a single start. The caught second-start failure diagnoses reuse of a terminated execution object; a new job run requires a new Thread.

</details>

### Complete the start and wait operations

Two campus welcome-desk assistants prepare materials. One job checks 4 badges and another checks 1 supply card. Each loop iteration records one completed check; these are model counts, not file operations.

Each CountJob owns its own total. Start both workers before joining either. Read and print only after both joins return normally.

Replace each START or WAIT marker with the taught Thread operation. Explain why construction alone does not schedule work, why both requests to start come before waiting, and why the caller can read each job result after its join returns normally. Predict all three printed lines before completing and running the program.

Read this supplied source, then put your complete solution in the Java editing cell after your response.

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(4);
CountJob second = new CountJob(1);
Thread firstWorker = new Thread(first);
Thread secondWorker = new Thread(second);
firstWorker.START();
secondWorker.START();
firstWorker.WAIT();
secondWorker.WAIT();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));
```

In [ ]:
Your response:


Record all three actual lines and compare them with your prediction. Identify which job owns each total. Explain how each normally returning join establishes completion and visibility before the caller reads that total. Distinguish the caller’s print order from worker finish order.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="guided-completion-answer-source-1"></a>

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(4);
CountJob second = new CountJob(1);
Thread firstWorker = new Thread(first);
Thread secondWorker = new Thread(second);
firstWorker.start();
secondWorker.start();
firstWorker.join();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));
```

Expected output:

```text
First: 4
Second: 1
Workers alive: false
```

<a id="guided-completion-interpretation"></a>

START is start: it requests each worker execution. WAIT is join: normal return establishes termination of that target and makes the worker's prior writes visible to the caller. The first job increments its private total four times and the second once. Both workers are terminated before isAlive is queried. The caller prints First before Second; this does not establish worker finish order or simultaneous execution.

</details>

### Change the amount of work

A desk supervisor redistributes the preparation work. The first assistant now has no items to check; the second has 4.

Use a fresh run of the original two-job program. Each worker still starts and is joined, including the zero-work job.

The supplied 3/2 program is the already explained baseline; do not predict that baseline again. Change only the two CountJob constructor limits to 0 and 4. Predict the new three lines and explain whether a zero-iteration job still needs the normal start/join lifecycle. Run the edited complete program and compare its actual result with your prediction.

Read this supplied source, then put your complete solution in the Java editing cell after your response.

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(3);
CountJob second = new CountJob(2);
Thread firstWorker = new Thread(first);
Thread secondWorker = new Thread(second);
firstWorker.start();
secondWorker.start();
firstWorker.join();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));
```

In [ ]:
Your response:


In [ ]:
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(3);
CountJob second = new CountJob(2);
Thread firstWorker = new Thread(first);
Thread secondWorker = new Thread(second);
firstWorker.start();
secondWorker.start();
firstWorker.join();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));

Record the three lines from the edited zero/four program. Explain how a job can be started and joined even though its loop has no iterations. Identify the normal join that makes each worker’s result available to the caller; preserve your earlier prediction.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="guided-modification-answer-source-1"></a>

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(0);
CountJob second = new CountJob(4);
Thread firstWorker = new Thread(first);
Thread secondWorker = new Thread(second);
firstWorker.start();
secondWorker.start();
firstWorker.join();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));
```

Expected output:

```text
First: 0
Second: 4
Workers alive: false
```

<a id="guided-modification-interpretation"></a>

The first job reaches its loop with index 0 and limit 0, so its body never runs and its initialized total stays 0. The second performs four increments. Zero work does not mean the Thread was never started: it starts, finishes and is joined like the other worker. Both joins establish completion before the caller reads either total.

</details>

### Repair a reused worker reference

A lab assistant needs a second preparation session after the first has ended. The first session checks 2 items; a fresh second CountJob should check 5.

The first worker has already terminated after its join. A Thread object may be started only once. Keep the two job objects separate.

This is a reading-only lifecycle diagnostic. Do not run the supplied draft. Trace which Thread object each worker variable refers to when secondWorker.start() is reached. Explain why the draft cannot complete that request and why creating second as a new CountJob alone is insufficient. Propose the single allowed line repair, then predict the repaired three report lines. Run only your complete repaired program.

Read this supplied source, then put your complete solution in the Java editing cell after your response.

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(2);
Thread firstWorker = new Thread(first);
firstWorker.start();
firstWorker.join();
CountJob second = new CountJob(5);
Thread secondWorker = firstWorker;
secondWorker.start();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));
```

In [ ]:
Your response:


Record the repaired program’s three lines. Trace the new Thread object to the second job and explain why the terminated first Thread cannot be started again. Explain what each normally returning join establishes. These sessions are sequential: state why their output does not demonstrate overlap.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="guided-debug-answer-source-1"></a>

```java
class CountJob implements Runnable {
    private int limit;
    private int total;
    public CountJob(int limit) { this.limit = limit; this.total = 0; }
    @Override
    public void run() {
        for (int index = 0; index < limit; index = index + 1) {
            total = total + 1;
        }
    }
    public int getTotal() { return total; }
}
CountJob first = new CountJob(2);
Thread firstWorker = new Thread(first);
firstWorker.start();
firstWorker.join();
CountJob second = new CountJob(5);
Thread secondWorker = new Thread(second);
secondWorker.start();
secondWorker.join();
System.out.println("First: " + first.getTotal());
System.out.println("Second: " + second.getTotal());
System.out.println("Workers alive: " + (firstWorker.isAlive() || secondWorker.isAlive()));
```

Expected output:

```text
First: 2
Second: 5
Workers alive: false
```

<a id="guided-debug-interpretation"></a>

The two worker variables in the draft refer to the same terminated Thread. A new work object does not replace that thread; starting its alias again violates the single-start lifecycle. Constructing new Thread(second) creates the required new execution path for the second job. The repaired second start/join completes five checks. The first remains 2, and both workers are terminated when reported. These sessions are deliberately sequential; the output does not demonstrate overlapping or parallel execution.

</details>

## Independent Practice

Build a complete program that transfers the taught mechanism to the following task. Keep the explained answer closed while planning, implementing and testing.

A campus organizer needs separate counts for names and supplies. Write WordCountJob as Runnable with words, minimum and a job-owned count. Count strings whose length is at least minimum. Use names Maya, Bo, Luis and supplies tea, café, water with minimum 4. Create one job and Thread for each array, start both before joining either, then print Names and Supplies counts from the caller. Plan the ownership and predict both lines before writing the program.

In [ ]:
Your response:


Record every actual baseline report. Compare it with your plan, explain the mechanism, and retain any discrepancy as evidence for a repair.

In [ ]:
Your response:


Before any boundary run, predict Names and Supplies for each fresh WordCountJob case with minimum 4: both arrays empty; one Maya and one café (exact threshold); names Bo/Li and supplies tea/cup (no matches). Keep the class, two starts, two joins and reports unchanged. Record why each case tests a different boundary.

In [ ]:
Your response:


Record the actual result for each named variant separately, preserving the predictions. Explain what boundary each checks and what it cannot establish about scheduling.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="independent-answer-source-1"></a>

```java
class WordCountJob implements Runnable {
    private String[] words;
    private int minimum;
    private int count;
    public WordCountJob(String[] words, int minimum) {
        this.words = words;
        this.minimum = minimum;
        this.count = 0;
    }
    @Override
    public void run() {
        for (String word : words) {
            if (word.length() >= minimum) { count = count + 1; }
        }
    }
    public int getCount() { return count; }
}
WordCountJob names = new WordCountJob(new String[]{"Maya", "Bo", "Luis"}, 4);
WordCountJob supplies = new WordCountJob(new String[]{"tea", "café", "water"}, 4);
Thread nameWorker = new Thread(names);
Thread supplyWorker = new Thread(supplies);
nameWorker.start();
supplyWorker.start();
nameWorker.join();
supplyWorker.join();
System.out.println("Names: " + names.getCount());
System.out.println("Supplies: " + supplies.getCount());
```

Expected output:

```text
Names: 2
Supplies: 2
```

<a id="independent-answer-source-2"></a>

```java
class WordCountJob implements Runnable {
    private String[] words;
    private int minimum;
    private int count;
    public WordCountJob(String[] words, int minimum) {
        this.words = words;
        this.minimum = minimum;
        this.count = 0;
    }
    @Override
    public void run() {
        for (String word : words) {
            if (word.length() >= minimum) { count = count + 1; }
        }
    }
    public int getCount() { return count; }
}
WordCountJob names = new WordCountJob(new String[]{}, 4);
WordCountJob supplies = new WordCountJob(new String[]{}, 4);
Thread nameWorker = new Thread(names);
Thread supplyWorker = new Thread(supplies);
nameWorker.start();
supplyWorker.start();
nameWorker.join();
supplyWorker.join();
System.out.println("Names: " + names.getCount());
System.out.println("Supplies: " + supplies.getCount());
```

Expected output:

```text
Names: 0
Supplies: 0
```

<a id="independent-answer-source-3"></a>

```java
class WordCountJob implements Runnable {
    private String[] words;
    private int minimum;
    private int count;
    public WordCountJob(String[] words, int minimum) {
        this.words = words;
        this.minimum = minimum;
        this.count = 0;
    }
    @Override
    public void run() {
        for (String word : words) {
            if (word.length() >= minimum) { count = count + 1; }
        }
    }
    public int getCount() { return count; }
}
WordCountJob names = new WordCountJob(new String[]{"Maya"}, 4);
WordCountJob supplies = new WordCountJob(new String[]{"café"}, 4);
Thread nameWorker = new Thread(names);
Thread supplyWorker = new Thread(supplies);
nameWorker.start();
supplyWorker.start();
nameWorker.join();
supplyWorker.join();
System.out.println("Names: " + names.getCount());
System.out.println("Supplies: " + supplies.getCount());
```

Expected output:

```text
Names: 1
Supplies: 1
```

<a id="independent-answer-source-4"></a>

```java
class WordCountJob implements Runnable {
    private String[] words;
    private int minimum;
    private int count;
    public WordCountJob(String[] words, int minimum) {
        this.words = words;
        this.minimum = minimum;
        this.count = 0;
    }
    @Override
    public void run() {
        for (String word : words) {
            if (word.length() >= minimum) { count = count + 1; }
        }
    }
    public int getCount() { return count; }
}
WordCountJob names = new WordCountJob(new String[]{"Bo", "Li"}, 4);
WordCountJob supplies = new WordCountJob(new String[]{"tea", "cup"}, 4);
Thread nameWorker = new Thread(names);
Thread supplyWorker = new Thread(supplies);
nameWorker.start();
supplyWorker.start();
nameWorker.join();
supplyWorker.join();
System.out.println("Names: " + names.getCount());
System.out.println("Supplies: " + supplies.getCount());
```

Expected output:

```text
Names: 0
Supplies: 0
```

<a id="independent-interpretation"></a>

Each WordCountJob owns its count and tests length >= minimum. Equal length counts; empty or below-threshold inputs do not. The caller joins both workers before reading either count. Caller print order does not reveal finish order.

</details>

## Summary

From memory, explain these lesson ideas in a connected account: Concurrent versus parallel execution, Work object versus execution path, Starting versus direct invocation, Single-start thread lifecycle, Interleaving and observable order, Join completion and visibility, Worker confinement. Include one limit of the example evidence.

In [ ]:
Your response:


<details>
<summary>Show answer</summary>

<a id="summary-retrieval-interpretation"></a>

Constructing work and its Thread does not start either. Use start for separate execution, and normal untimed joins on started workers before reading their results. Keep each worker's changing state separate. Interleaving can vary; parallel execution and finish order are not promised. A Thread has one start, and isAlive needs lifecycle context.

</details>

## Reflection

Describe a campus or project task that could use this lesson’s mechanism. Identify the work, owned or shared state, completion rule and one limitation of the analogy. Explain what would fail if the rule were omitted.

In [ ]:
Your response:


The next lesson studies state that both workers change. Separate owned results are no longer enough when the same value must be protected.

## Supplemental Reading

- [Java 21 Thread](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Thread.html) documents construction, start, direct run, join, currentThread, and isAlive.
- [Java 21 Runnable](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Runnable.html) defines the work method implemented by a job.
- [Java language memory ordering](https://docs.oracle.com/javase/specs/jls/se21/html/jls-17.html#jls-17.4.5) specifies the visibility relationship established by a completed join.
- [Defining and starting a thread](https://docs.oracle.com/javase/tutorial/essential/concurrency/runthread.html) is the assigned Oracle tutorial reading on work and execution paths. Its tutorial examples target an older Java release; the Java 21 API links above describe our runtime.